In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)

In [4]:
train=pd.concat([pd.read_csv('SDNFlow_Dataset_Normal.csv'),pd.read_csv('SDNFlow_Dataset_attack.csv')])

In [5]:
train=train.drop(columns=['ip_proto','src_port','dst_port','flow_ID','eth_type','ipv4_src','ipv4_dst','TnBPDstIP','TnPPDstIP','TnBPSrcIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP'])

In [6]:
train.head()

,flow_duration,tot_len_pkts,tot_pkts,flow_byts_s,flow_pkts_s,pkt_size_average,fwd_header_len,Byts_max_interval_2s,Byts_min_interval_2s,Byts_mean_interval_2s,Byts_std_interval_2s,Pkts_max_interval_2s,Pkts_mean_interval_2s,Pkts_min_interval_2s,Pkts_std_interval_2s,active_max,active_mean,active_min,active_std,idle_max,idle_mean,idle_min,idle_std,Attack,Category
0,6.472,113472,628,17532.756490,97.033375,180.687898,12560,112694,0,28368.00000,48686.674830,625,157.000000,0,270.202702,2,2.0,2,0.0,5,5.0,5,0.0,0,SSH
1,6.465,24492,370,3788.399072,57.231245,66.194595,7400,24294,0,6123.00000,10491.343150,367,92.500000,0,158.487381,2,2.0,2,0.0,5,5.0,5,0.0,0,SSH
2,4.798,2211,4,460.817007,0.833681,552.750000,32,1180,0,737.00000,524.675773,2,1.333333,0,0.942809,4,4.0,4,0.0,0,0.0,0,0.0,0,ICMP
3,595.270,10827817,25627,18189.757590,43.051052,422.515979,205016,44867,11454,36334.95638,2870.840349,103,85.996644,22,4.407512,596,596.0,596,0.0,0,0.0,0,0.0,0,STREAMING
4,595.251,154551644,114704,259641.132900,192.698542,1347.395418,917632,935885,32828,518629.67790,121204.950600,673,384.912752,29,84.666811,596,596.0,596,0.0,0,0.0,0,0.0,0,STREAMING


In [7]:
from sklearn.preprocessing import LabelEncoder

In [8]:
le=LabelEncoder()

In [9]:
train['Category']=le.fit_transform(train['Category'])

In [10]:
train['Category'].value_counts()

Category
2     164974
10    151456
8     123807
7      74644
9      67288
0      53551
11     17787
12      6282
6       1487
5       1133
3        305
1         91
4         23
Name: count, dtype: int64

In [11]:
Y=train['Category']

In [12]:
X=train

In [13]:
X.shape

(662828, 25)

In [14]:
Y.shape

(662828,)

In [15]:
X.head()

,flow_duration,tot_len_pkts,tot_pkts,flow_byts_s,flow_pkts_s,pkt_size_average,fwd_header_len,Byts_max_interval_2s,Byts_min_interval_2s,Byts_mean_interval_2s,Byts_std_interval_2s,Pkts_max_interval_2s,Pkts_mean_interval_2s,Pkts_min_interval_2s,Pkts_std_interval_2s,active_max,active_mean,active_min,active_std,idle_max,idle_mean,idle_min,idle_std,Attack,Category
0,6.472,113472,628,17532.756490,97.033375,180.687898,12560,112694,0,28368.00000,48686.674830,625,157.000000,0,270.202702,2,2.0,2,0.0,5,5.0,5,0.0,0,5
1,6.465,24492,370,3788.399072,57.231245,66.194595,7400,24294,0,6123.00000,10491.343150,367,92.500000,0,158.487381,2,2.0,2,0.0,5,5.0,5,0.0,0,5
2,4.798,2211,4,460.817007,0.833681,552.750000,32,1180,0,737.00000,524.675773,2,1.333333,0,0.942809,4,4.0,4,0.0,0,0.0,0,0.0,0,3
3,595.270,10827817,25627,18189.757590,43.051052,422.515979,205016,44867,11454,36334.95638,2870.840349,103,85.996644,22,4.407512,596,596.0,596,0.0,0,0.0,0,0.0,0,6
4,595.251,154551644,114704,259641.132900,192.698542,1347.395418,917632,935885,32828,518629.67790,121204.950600,673,384.912752,29,84.666811,596,596.0,596,0.0,0,0.0,0,0.0,0,6


In [16]:
X=X.drop(columns='Category')

In [17]:
from sklearn.model_selection import train_test_split

In [18]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [19]:
from xgboost import XGBClassifier
     

xgb=XGBClassifier()
     

estimators=[('xgb',xgb)]
     

from sklearn.pipeline import Pipeline
     

pipe=Pipeline(steps=estimators)
     

pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None


In [20]:
from skopt import BayesSearchCV
     

from skopt.space import Real,Categorical,Integer
     

search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}
    

In [21]:
opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=8)
     

opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3
)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

opt.fit(X_train, Y_train)

Y_pred_train = opt.predict(X_train)
Y_pred_test = opt.predict(X_test)

print("Training Accuracy:", accuracy_score(Y_train, Y_pred_train))
print("Testing Accuracy:", accuracy_score(Y_test, Y_pred_test))

print("Test Confusion Matrix:")
print(confusion_matrix(Y_test, Y_pred_test))

print("Test Classification Report:")
print(classification_report(Y_test, Y_pred_test))

In [ ]:
test=pd.read_csv('modified_concatenated_InSDN_DatasetCSV.csv')

In [ ]:
test.head()

,Flow ID,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.3.130-200.175.2.130-38694-4444-6,10/1/2020 5:02,269709,4,5,48,23,30,0,12.0,14.696938,23,0,4.600000,10.285913,263.246684,33.369298,33713.625000,90272.549880,257068,14,260808,86936.0,149854.765700,259973,104,269709,67427.250000,126912.740800,257785,2956,0,0,0,0,128,176,14.830799,18.538499,0,30,7.100000,11.779926,138.766667,0,1,0,0,0,0,0,0,1,7.888889,12.0,4.600000,0,0,0,0,0,0,4,48,5,23,-1,64,2,0,0.0,0.0,0,0,0.0,0.0,0,0,U2R
1,192.168.3.130-200.175.2.130-38693-4444-6,10/1/2020 5:02,268599,2,3,0,23,0,0,0.0,0.000000,23,0,7.666667,13.279056,85.629507,18.615110,67149.750000,132430.553600,265786,11,265811,265811.0,0.000000,265811,265811,268574,134287.000000,185983.225600,265797,2777,0,0,0,0,64,112,7.446044,11.169066,0,23,3.833333,9.389711,88.166667,0,1,0,0,0,0,0,0,1,4.600000,0.0,7.666667,0,0,0,0,0,0,2,0,3,23,-1,64,0,0,0.0,0.0,0,0,0.0,0.0,0,0,U2R
2,192.168.3.130-200.175.2.130-3632-33747-6,10/1/2020 5:02,22194,5,5,53,30,30,0,10.6,14.724130,30,0,6.000000,13.416408,3739.749482,450.572227,2466.000000,2951.053583,8063,3,17518,4379.5,6184.320254,13537,3,15128,3782.000000,3420.925898,8063,42,0,0,0,0,160,176,225.286113,225.286113,0,30,7.545455,13.048859,170.272727,0,1,0,0,0,0,0,0,1,8.300000,10.6,6.000000,0,0,0,0,0,0,5,53,5,30,-1,215,2,0,0.0,0.0,0,0,0.0,0.0,0,0,U2R
3,192.168.3.130-200.175.2.130-8180-38745-6,10/1/2020 1:39,9556,4,4,30,30,30,0,7.5,15.000000,30,0,7.500000,15.000000,6278.777731,837.170364,1365.142857,1447.714225,4022,13,5511,1837.0,1441.824885,3380,524,5588,1862.666667,2386.259067,4559,23,0,0,0,0,128,144,418.585182,418.585182,0,30,6.666667,13.228757,175.000000,0,1,0,0,0,0,0,0,1,7.500000,7.5,7.500000,0,0,0,0,0,0,4,30,4,30,-1,215,1,0,0.0,0.0,0,0,0.0,0.0,0,0,BFA
4,192.168.3.130-200.175.2.130-8180-37217-6,10/1/2020 1:39,8782,4,4,30,30,30,0,7.5,15.000000,30,0,7.500000,15.000000,6832.156684,910.954225,1254.571429,1607.096435,4049,8,4725,1575.0,2277.042160,4204,226,5618,1872.666667,2191.816674,4287,8,0,0,0,0,128,144,455.477112,455.477112,0,30,6.666667,13.228757,175.000000,0,1,0,0,0,0,0,0,1,7.500000,7.5,7.500000,0,0,0,0,0,0,4,30,4,30,-1,215,1,0,0.0,0.0,0,0,0.0,0.0,0,0,BFA


In [ ]:
test.columns

Index(['Flow ID', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
       'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max',
       'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags',
       'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s',
       'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean',
       'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt',
       'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt',
       'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg',
       'Fwd Seg Siz

In [ ]:
test=test.drop(columns=['Flow ID','Timestamp','Flow Duration'])

In [ ]:
test['Label']=le.fit_transform(test['Label'])

In [ ]:
B=test['Label']

In [ ]:
A=test

In [ ]:
A=A.drop(columns=['Label'])

In [ ]:
A.shape

(343889, 75)

In [ ]:
B.shape

(343889,)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
A_train,A_test,B_train,B_test=train_test_split(A,B,test_size=0.2,random_state=42)

In [ ]:
X_train.head()

,flow_duration,tot_len_pkts,tot_pkts,flow_byts_s,flow_pkts_s,pkt_size_average,fwd_header_len,Byts_max_interval_2s,Byts_min_interval_2s,Byts_mean_interval_2s,Byts_std_interval_2s,Pkts_max_interval_2s,Pkts_mean_interval_2s,Pkts_min_interval_2s,Pkts_std_interval_2s,active_max,active_mean,active_min,active_std,idle_max,idle_mean,idle_min,idle_std,Attack
4488,0.347,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1
46368,0.775,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1
325456,6.242,234,4,37.487985,0.640820,58.5,80,126,0,58.5,44.774435,2,1.0,0,0.707107,4,3.0,2,1.0,3,3.0,3,0.0,1
125553,6.449,576,4,89.316173,0.620251,144.0,80,444,0,144.0,175.288334,2,1.0,0,0.707107,4,3.0,2,1.0,3,3.0,3,0.0,1
58157,0.575,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1


In [ ]:
A_train.head()

,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
260698,0,2,0,0,0,0,0.0,0.000000,0,0,0.000000,0.000000,0.000000,715.307582,2796.000,0.000000e+00,2796,2796,0,0.0,0.000000,0,0,2796,2796.0,0.0,2796,2796,0,0,0,0,0,40,0.000000,715.307582,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,0,0,0,0.0,0.0,0,0,0.0,0.0,0,0
71472,0,2,0,0,0,0,0.0,0.000000,0,0,0.000000,0.000000,0.000000,39215.686270,51.000,0.000000e+00,51,51,0,0.0,0.000000,0,0,51,51.0,0.0,51,51,0,0,0,0,0,0,0.000000,39215.686270,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,-1,0,0,0.0,0.0,0,0,0.0,0.0,0,0
100803,3,7,30,30,30,0,10.0,17.320508,30,0,4.285714,11.338934,0.951526,0.158588,7006289.222,2.100000e+07,63000000,10,1788,894.0,1090.358657,1665,123,63100000,10500000.0,25700000.0,63000000,263,0,0,0,0,104,232,0.047576,0.111011,0,30,5.454545,12.135598,147.272727,1,0,0,0,1,0,0,0,2,6.0,10.0,4.285714,0,0,0,0,0,0,3,30,7,30,-1,63,1,0,3931.0,0.0,3931,3931,63000000.0,0.0,63000000,63000000
1012,0,2,0,0,0,0,0.0,0.000000,0,0,0.000000,0.000000,0.000000,33898.305080,59.000,0.000000e+00,59,59,0,0.0,0.000000,0,0,59,59.0,0.0,59,59,0,0,0,0,0,0,0.000000,33898.305080,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,-1,0,0,0.0,0.0,0,0,0.0,0.0,0,0
288044,2,1,0,0,0,0,0.0,0.000000,0,0,0.000000,0.000000,0.000000,723.414516,2073.500,2.930958e+03,4146,1,4146,4146.0,0.000000,4146,4146,0,0.0,0.0,0,0,0,0,0,0,40,20,482.276344,241.138172,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,2,0,1,0,-1,64240,0,0,0.0,0.0,0,0,0.0,0.0,0,0


In [ ]:
A_train=A_train.drop(columns=[
    'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 
    'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 
    'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 
    'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 
    'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 
    'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 
    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 
    'Fwd Header Len', 'Bwd Header Len'
])

In [ ]:
A_train=A_train.rename(columns={'Tot Fwd Pkts':'tot_pkts','TotLen Fwd Pkts':'tot_len_pkts'})

In [ ]:
A_train.head()

,tot_pkts,Tot Bwd Pkts,tot_len_pkts,TotLen Bwd Pkts,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
260698,0,2,0,0,0.000000,715.307582,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,0,0,0,0.0,0.0,0,0,0.0,0.0,0,0
71472,0,2,0,0,0.000000,39215.686270,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,-1,0,0,0.0,0.0,0,0,0.0,0.0,0,0
100803,3,7,30,30,0.047576,0.111011,0,30,5.454545,12.135598,147.272727,1,0,0,0,1,0,0,0,2,6.0,10.0,4.285714,0,0,0,0,0,0,3,30,7,30,-1,63,1,0,3931.0,0.0,3931,3931,63000000.0,0.0,63000000,63000000
1012,0,2,0,0,0.000000,33898.305080,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,0,0,2,0,-1,-1,0,0,0.0,0.0,0,0,0.0,0.0,0,0
288044,2,1,0,0,482.276344,241.138172,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,0.0,0.0,0.000000,0,0,0,0,0,0,2,0,1,0,-1,64240,0,0,0.0,0.0,0,0,0.0,0.0,0,0


In [ ]:
A_train = A_train.drop(columns=[
    'Tot Bwd Pkts',
    'TotLen Bwd Pkts',
    'Pkt Len Min',
    'Pkt Len Max',
    'Pkt Len Mean',
    'Pkt Len Std',
    'Pkt Len Var',
    'FIN Flag Cnt',
    'SYN Flag Cnt',
    'RST Flag Cnt',
    'PSH Flag Cnt',
    'ACK Flag Cnt',
    'URG Flag Cnt',
    'CWE Flag Count',
    'ECE Flag Cnt',
    'Down/Up Ratio',
    'Pkt Size Avg',
    'Fwd Seg Size Avg',
    'Bwd Seg Size Avg',
    'Fwd Byts/b Avg',
    'Fwd Pkts/b Avg',
    'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg',
    'Bwd Pkts/b Avg',
    'Bwd Blk Rate Avg',
    'Subflow Fwd Pkts',
    'Subflow Fwd Byts',
    'Subflow Bwd Pkts',
    'Subflow Bwd Byts',
    'Init Fwd Win Byts',
    'Init Bwd Win Byts',
    'Fwd Act Data Pkts',
    'Fwd Seg Size Min'
])

In [ ]:
X_train.head()

,flow_duration,tot_len_pkts,tot_pkts,flow_byts_s,flow_pkts_s,pkt_size_average,fwd_header_len,Byts_max_interval_2s,Byts_min_interval_2s,Byts_mean_interval_2s,Byts_std_interval_2s,Pkts_max_interval_2s,Pkts_mean_interval_2s,Pkts_min_interval_2s,Pkts_std_interval_2s,active_max,active_mean,active_min,active_std,idle_max,idle_mean,idle_min,idle_std,Attack
4488,0.347,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1
46368,0.775,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1
325456,6.242,234,4,37.487985,0.640820,58.5,80,126,0,58.5,44.774435,2,1.0,0,0.707107,4,3.0,2,1.0,3,3.0,3,0.0,1
125553,6.449,576,4,89.316173,0.620251,144.0,80,444,0,144.0,175.288334,2,1.0,0,0.707107,4,3.0,2,1.0,3,3.0,3,0.0,1
58157,0.575,0,0,0.000000,0.000000,0.0,0,0,0,0.0,0.000000,0,0.0,0,0.000000,1,1.0,1,0.0,0,0.0,0,0.0,1


In [ ]:
A_train.head()

,tot_pkts,tot_len_pkts,Fwd Pkts/s,Bwd Pkts/s,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
260698,0,0,0.000000,715.307582,0.0,0.0,0,0,0.0,0.0,0,0
71472,0,0,0.000000,39215.686270,0.0,0.0,0,0,0.0,0.0,0,0
100803,3,30,0.047576,0.111011,3931.0,0.0,3931,3931,63000000.0,0.0,63000000,63000000
1012,0,0,0.000000,33898.305080,0.0,0.0,0,0,0.0,0.0,0,0
288044,2,0,482.276344,241.138172,0.0,0.0,0,0,0.0,0.0,0,0


In [ ]:
A_train=A_train.rename(columns={'Fwd Pkts/s':'flow_pkts_s'})

In [ ]:
A_train = A_train.rename(columns={
    'Active Mean': 'active_mean',
    'Active Std': 'active_std',
    'Active Max': 'active_max',
    'Active Min': 'active_min',
    'Idle Mean': 'idle_mean',
    'Idle Std': 'idle_std',
    'Idle Max': 'idle_max',
    'Idle Min': 'idle_min'
})


In [ ]:
A_train.head()

,tot_pkts,tot_len_pkts,flow_pkts_s,Bwd Pkts/s,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
260698,0,0,0.000000,715.307582,0.0,0.0,0,0,0.0,0.0,0,0
71472,0,0,0.000000,39215.686270,0.0,0.0,0,0,0.0,0.0,0,0
100803,3,30,0.047576,0.111011,3931.0,0.0,3931,3931,63000000.0,0.0,63000000,63000000
1012,0,0,0.000000,33898.305080,0.0,0.0,0,0,0.0,0.0,0,0
288044,2,0,482.276344,241.138172,0.0,0.0,0,0,0.0,0.0,0,0


In [ ]:
A_train=A_train.drop(columns='Bwd Pkts/s')

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


B_pred_train = opt.predict(A_train)
B_pred_test = opt.predict(A_test)

print("Training Accuracy:", accuracy_score(B_train, B_pred_train))
print("Testing Accuracy:", accuracy_score(B_test, B_pred_test))

print("Test Confusion Matrix:")
print(confusion_matrix(B_test, B_pred_test))

print("Test Classification Report:")
print(classification_report(B_test, B_pred_test))